In [12]:
import ssl
import certifi
import urllib.request
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optimimport 
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from PIL import Image
import torch.optim as optim

In [13]:
torch.hub.set_dir('/tmp/torch_hub')

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(512, 2)

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [16]:
training_set = ImageFolder('/Users/aymanaghel/Desktop/Crater CV/moon_photos', transform=transform)
print(training_set.classes)
print(len(training_set))

testing_set = ImageFolder('/Users/aymanaghel/Desktop/Crater CV/mars_photos', transform=transform)
print(testing_set.classes)
print(len(testing_set))

['moon_crater', 'moon_non']
6030
['mars_crater', 'mars_non']
1200


In [17]:
training_size = 0.8 * len(training_set)
testing_size = len(training_set) - training_size

print(testing_size)
print(training_size / testing_size)

1206.0
4.0


In [18]:
from torch.utils.data import random_split

train_size = int(0.8 * len(training_set))
test_size = len(training_set) - train_size

train_dataset, test_dataset = random_split(training_set, [train_size, test_size])

print(train_size, test_size)

4824 1206


In [23]:
train_loader = DataLoader(training_set, batch_size=32, shuffle=True)
test_loader = DataLoader(testing_set, batch_size=32, shuffle=False)

In [20]:
for epoch in range(7):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.0453
Epoch 2, Loss: 0.0039
Epoch 3, Loss: 0.0024
Epoch 4, Loss: 0.0012
Epoch 5, Loss: 0.0009
Epoch 6, Loss: 0.0013
Epoch 7, Loss: 0.0014


In [21]:
torch.save(model.state_dict(), 'craters.pth')

In [24]:
model.load_state_dict(torch.load('craters.pth', map_location='cpu'))

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"{100 * correct / total:2f}% accuracy")


51.083333% accuracy
